In [1]:
# ─── 1. CÀI ĐẶT MÔI TRƯỜNG ───────────────────────────────────────────────────
!pip uninstall -y transformers peft accelerate -q
!pip install -q \
    transformers==4.41.2 \
    peft==0.11.1 \
    accelerate==0.30.1 \
    datasets \
    scikit-learn \
    sentencepiece \
    safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.5 MB/s eta 0:00:00


In [ ]:


# ─── 2. SETUP & IMPORTS ───────────────────────────────────────────────────────
import os, time, functools, json
import torch
import numpy as np
import torch.nn as nn
from PIL import Image
from datasets import load_dataset, Image as HFImage
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (CLIPProcessor, CLIPModel, TrainingArguments,
                          Trainer, EarlyStoppingCallback)
from transformers.modeling_outputs import SequenceClassifierOutput
from google.colab import drive

# Tối ưu bộ nhớ GPU
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Kết nối Drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/CLIP_PromptTuning_IFND_Final"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Save to:", SAVE_DIR)

# ─── 3. DATASET & PREPROCESS (IFND) ──────────────────────────────────────────
print("⏳ Loading dataset IFND-multimodal...")
dataset = load_dataset("Nhat243/IFND-multimodal")
dataset = dataset.cast_column("image", HFImage(decode=True))

model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)
PROMPT_LENGTH = 16 # Tăng lên 16 cho IFND để học sâu hơn

def preprocess(example):
    try:
        image = example['image'].convert("RGB")
    except:
        image = Image.new("RGB", (224, 224), (128, 128, 128))

    inputs = processor(
        text=str(example['text']),
        images=image,
        padding="max_length",
        truncation=True,
        max_length=77 - PROMPT_LENGTH,
        return_tensors="pt"
    )
    return {
        "input_ids": inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values": inputs["pixel_values"][0],
        "labels": int(example["label"])
    }

print("⏳ Preprocessing dataset IFND...")
encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names,
    desc="Preprocessing"
)

encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

# ─── 4. MODEL ARCHITECTURE (PROMPT TUNING - MIRAGE STYLE) ─────────────────────
class CLIPPromptTuningIFND(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32", num_labels=2,
                 prompt_length=16, lambda_weight=0.1, delta=0.5):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(model_name)
        self.prompt_length = prompt_length

        # 1. Đóng băng Base Model
        for param in self.clip.parameters():
            param.requires_grad = False

        self.hidden_dim = self.clip.config.text_config.hidden_size
        embed_dim = self.clip.config.projection_dim # 512

        # 2. Khởi tạo Soft Prompt (Học được)
        self.soft_prompt = nn.Parameter(torch.randn(prompt_length, self.hidden_dim))

        # 3. Gated Fusion & Alignment Head
        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)
        self.W_g = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

        # Mở khóa Gradient cho các thành phần học
        self.soft_prompt.requires_grad = True
        for param in self.cross_attention.parameters(): param.requires_grad = True
        for param in self.W_g.parameters(): param.requires_grad = True
        for param in self.classifier.parameters(): param.requires_grad = True

        self.lambda_weight = lambda_weight
        self.delta = delta
        self.align_loss_fn = nn.CosineEmbeddingLoss(margin=delta)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        batch_size = input_ids.size(0)
        device = input_ids.device

        # --- TEXT PATHWAY (Sử dụng Encoder trực tiếp để tránh lỗi TypeError) ---
        token_embeddings = self.clip.text_model.embeddings.token_embedding(input_ids)
        soft_prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        hidden_states = torch.cat([soft_prompt, token_embeddings], dim=1) # Length: 77

        prompt_mask = torch.ones(batch_size, self.prompt_length, device=device)
        attention_mask_text = torch.cat([prompt_mask, attention_mask], dim=1)

        # Xử lý Position Embeddings thủ công cho chuỗi 77
        seq_length = hidden_states.size(1)
        position_ids = torch.arange(seq_length, device=device).unsqueeze(0)
        position_embeddings = self.clip.text_model.embeddings.position_embedding(position_ids)
        hidden_states = hidden_states + position_embeddings

        # Tạo Causal Mask bắt buộc cho CLIP Text
        causal_attention_mask = torch.full((seq_length, seq_length), float("-inf"), device=device)
        causal_attention_mask = torch.triu(causal_attention_mask, diagonal=1)
        causal_attention_mask = causal_attention_mask.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, seq_length, seq_length)

        if attention_mask_text is not None:
            # Mask format: [batch, 1, seq, seq]
            attention_mask_text = (1.0 - attention_mask_text[:, None, None, :].expand(batch_size, 1, seq_length, seq_length)) * torch.finfo(hidden_states.dtype).min

        encoder_outputs = self.clip.text_model.encoder(
            inputs_embeds=hidden_states,
            attention_mask=attention_mask_text,
            causal_attention_mask=causal_attention_mask,
        )

        hidden_states = encoder_outputs[0]
        hidden_states = self.clip.text_model.final_layer_norm(hidden_states)

        # Lấy EOS token (pooler output) - Argmax tìm vị trí EOS gốc + offset prompt
        eos_positions = input_ids.argmax(dim=1) + self.prompt_length
        pooled_output = hidden_states[torch.arange(batch_size, device=device), eos_positions]

        text_features = self.clip.text_projection(pooled_output)
        h_T = text_features / text_features.norm(dim=-1, keepdim=True)

        # --- IMAGE PATHWAY ---
        image_outputs = self.clip.vision_model(pixel_values=pixel_values)
        image_features = self.clip.visual_projection(image_outputs.pooler_output)
        h_I = image_features / image_features.norm(dim=-1, keepdim=True)

        # --- GATED MODALITY FUSION ---
        attn_out, _ = self.cross_attention(h_T.unsqueeze(1), h_I.unsqueeze(1), h_I.unsqueeze(1))
        z = attn_out.squeeze(1)
        gate = torch.sigmoid(self.W_g(torch.cat([h_T, h_I], dim=1)))
        h_f = gate * z

        # --- LOSS ---
        logits = self.classifier(h_f)
        loss = None
        if labels is not None:
            loss_cls = nn.CrossEntropyLoss()(logits, labels)
            # Khớp Align Loss IFND: labels * 2 - 1
            loss_align = self.align_loss_fn(h_T, h_I, labels.float() * 2 - 1)
            loss = loss_cls + (self.lambda_weight * loss_align)

        return SequenceClassifierOutput(loss=loss, logits=logits)

import gc; gc.collect(); torch.cuda.empty_cache()
model = CLIPPromptTuningIFND(prompt_length=PROMPT_LENGTH).to(device)

# ─── 5. TRAINING ─────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds), "f1": f1, "auc": roc_auc_score(labels, probs[:, 1])}

training_args = TrainingArguments(
    output_dir=SAVE_DIR, num_train_epochs=5, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, learning_rate=1e-3, evaluation_strategy="epoch",
    save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1",
    fp16=True, report_to="none"
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(2)]
)

print("🚀 Huấn luyện Prompt Tuning...")
start_train = time.time()
trainer.train()
train_time = (time.time() - start_train) / 60

# ─── 6. LATENCY & TEST EVALUATION ───────────────────────────────────────────
model.eval()
dummy_img = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_txt = "Fake news detection sample."
inputs = processor(text=dummy_txt, images=dummy_img, truncation=True, padding="max_length", max_length=77-PROMPT_LENGTH, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

latencies = []
with torch.no_grad():
    for _ in range(50): model(**inputs) # Warmup
    for _ in range(200):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(**inputs)
        torch.cuda.synchronize()
        latencies.append((time.perf_counter()-t0)*1000)

vram = torch.cuda.max_memory_allocated() / 1024**3
test_res = trainer.predict(encoded_dataset["test"])
m = test_res.metrics

final_results = {
    "Model": "CLIP ViT-B/32", "Method": "Prompt Tuning (Length=16)", "Dataset": "IFND",
    "Hardware_Stats": {
        "Training_Time_Min": round(train_time, 2),
        "Peak_VRAM_GB": round(vram, 2),
        "Latency_P50_ms": round(np.median(latencies), 2)
    },
    "Test_Metrics": {
        "Accuracy": round(m['test_accuracy']*100, 2),
        "F1_Score": round(m['test_f1']*100, 2),
        "AUC": round(m['test_auc'], 4)
    }
}

with open(os.path.join(SAVE_DIR, "results_PromptTuning_IFND.json"), "w") as f:
    json.dump(final_results, f, indent=4)

print(f"✅ Xong! F1: {final_results['Test_Metrics']['F1_Score']}%")
from google.colab import runtime
runtime.unassign()

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save to: /content/drive/MyDrive/CLIP_PromptTuning_IFND_Final
⏳ Loading dataset IFND-multimodal...
⏳ Preprocessing dataset IFND...


Preprocessing:   0%|          | 0/8416 [00:00<?, ? examples/s]

Preprocessing:   0%|          | 0/1052 [00:00<?, ? examples/s]

Preprocessing:   0%|          | 0/1053 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


🚀 Huấn luyện Prompt Tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Auc
1,0.344800,0.143609,0.970532,0.980923,0.990798
2,0.132700,0.118283,0.980038,0.987045,0.995219
3,0.112400,0.112451,0.978137,0.985794,0.995753
4,0.094700,0.109736,0.982890,0.988820,0.996270
5,0.091100,0.108291,0.983840,0.989487,0.996459


✅ Xong! F1: 99.26%
